In [1]:
import pandas as pd
import polars as pl
from matplotlib import pyplot as plt
import plotnine as pn

In [2]:
import sys
sys.path.append("../")   # path to folder containing the python file

from utils.load_gtf_cgc_dresden import *
from ProteinExpression.load_pr_data import *
from utils.util_functions import *

In [32]:
mapping_table = pd.read_csv("./random_mapping.csv")

sa = pd.read_csv("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/sample_data/master_drop_sample_annotation_sizeFactorFiltered_0.1.tsv", sep="\t")


In [6]:
spliceosome_genes = pd.read_csv("https://www.genenames.org/cgi-bin/genegroup/download?id=1518&type=branch", sep = "\t")


In [62]:
needed_cols = ["#CHROM", "POS", "REF", "ALT", "germline_Consequence", "germline_IMPACT",  "SpliceAI_pred_DS_AL", "SpliceAI_pred_DS_AG", 
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL", "germline_max_spliceai_score", "PID",  "SYMBOL", "Gene"]


dresden_germline_splice = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/germline/germline_damaging_aggregated_all.parquet")
                .filter(pl.col("Gene").is_in(spliceosome_genes["Ensembl gene ID"]))
                .select(needed_cols) 
                 
                .collect(engine="streaming")        
                  ).to_pandas()


dresden_germline_splice = dresden_germline_splice[(dresden_germline_splice["germline_Consequence"].isin(["splice_donor_variant", "splice_acceptor_variant"])) | (dresden_germline_splice["germline_max_spliceai_score"] >= 0.2)]



In [63]:

needed_cols = ["#CHROM", "POS", "REF", "ALT", "somatic_Consequence", "somatic_IMPACT",  "SpliceAI_pred_DS_AL", "SpliceAI_pred_DS_AG", 
    "SpliceAI_pred_DS_DG", "SpliceAI_pred_DS_DL", "somatic_max_spliceai_score", "PID",  "SYMBOL", "Gene"]


dresden_somatic_splice = (pl.scan_parquet("/omics/odcf/analysis/hipo/hipo_021/outlier_analysis/vcf/dresden_annotations/somatic/somatic_damaging_aggregated_all.parquet")
                .filter(pl.col("Gene").is_in(spliceosome_genes["Ensembl gene ID"]))
                .select(needed_cols) 
                 
                .collect(engine="streaming")        
                  ).to_pandas()


dresden_somatic_splice = dresden_somatic_splice[(dresden_somatic_splice["somatic_Consequence"].isin(["splice_donor_variant", "splice_acceptor_variant"])) | (dresden_germline_splice["germline_max_spliceai_score"] >= 0.2)]




/tmp/ipykernel_2841646/189342711.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.


In [64]:
dresden_somatic_splice = dresden_somatic_splice.merge(mapping_table, left_on="PID", right_on="nct_id").drop(columns=["PID", "nct_id",])

somatic_counts = dresden_somatic_splice.groupby(['Gene', 'random_id']).size().reset_index(name='n_somatic_splicing_variant')
somatic_counts

,Gene,random_id,n_somatic_splicing_variant
0,ENSG00000003756,BRI3662,1
1,ENSG00000003756,EAA9516,1
2,ENSG00000003756,HIY6488,1
3,ENSG00000003756,HJW6671,2
4,ENSG00000003756,PYZ9132,1
...,...,...,...
502,ENSG00000240682,LNG5339,1
503,ENSG00000240682,VAP5984,1
504,ENSG00000257103,ABV3837,1
505,ENSG00000257103,DIB5249,1


In [65]:
dresden_germline_splice = dresden_germline_splice.merge(mapping_table, left_on="PID", right_on="nct_id").drop(columns=["PID", "nct_id"])

germline_counts = dresden_germline_splice.groupby(['Gene', 'random_id']).size().reset_index(name='n_germline_splicing_variant')
germline_counts

,Gene,random_id,n_germline_splicing_variant
0,ENSG00000003756,AHY8852,1
1,ENSG00000003756,CFD2833,1
2,ENSG00000003756,CXI8998,1
3,ENSG00000003756,DCU0059,1
4,ENSG00000003756,FAV0278,1
...,...,...,...
863,ENSG00000257103,BNP9147,1
864,ENSG00000257103,BNR4176,2
865,ENSG00000257103,CJS5224,1
866,ENSG00000257103,NTF5010,1


In [72]:
germline_counts.merge(somatic_counts, on=["Gene", "random_id"], how="outer").fillna(0)

,Gene,random_id,n_germline_splicing_variant,n_somatic_splicing_variant
0,ENSG00000003756,AHY8852,1.0,0.0
1,ENSG00000003756,BRI3662,0.0,1.0
2,ENSG00000003756,CFD2833,1.0,0.0
3,ENSG00000003756,CXI8998,1.0,0.0
4,ENSG00000003756,DCU0059,1.0,0.0
...,...,...,...,...
1370,ENSG00000257103,CJS5224,1.0,0.0
1371,ENSG00000257103,DIB5249,0.0,1.0
1372,ENSG00000257103,EIZ6791,0.0,1.0
1373,ENSG00000257103,NTF5010,1.0,0.0


In [69]:
g

<ArrowStringArray>
['ENSG00000003756', 'ENSG00000018610', 'ENSG00000021776', 'ENSG00000054118',
 'ENSG00000060339', 'ENSG00000060688', 'ENSG00000063244', 'ENSG00000067596',
 'ENSG00000068394', 'ENSG00000076650',
 ...
 'ENSG00000185324', 'ENSG00000189091', 'ENSG00000196504', 'ENSG00000197451',
 'ENSG00000198563', 'ENSG00000198783', 'ENSG00000204560', 'ENSG00000240344',
 'ENSG00000240682', 'ENSG00000257103']
Length: 124, dtype: str